### Data Ingestion

In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is a test document.",
    metadata={
        "source": "test_source.txt", "pages": 1,
        "author": "John Doe", 
        "date_created": "2024-06-01"
    },
)
doc

Document(metadata={'source': 'test_source.txt', 'pages': 1, 'author': 'John Doe', 'date_created': '2024-06-01'}, page_content='This is a test document.')

In [3]:
## Create a simple text file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_texts = {
    # Path to Python language documentation file
    "../data/text_files/python.txt": """Python is a high-level, interpreted programming language.
- Easy to learn and read with clean syntax
- Versatile: web development, data science, AI, automation
- Extensive standard library and third-party packages
- Dynamic typing with strong type inference
- Cross-platform support (Windows, Linux, macOS)
- Large community and excellent documentation
- Supports multiple programming paradigms""",
    # Path to C language documentation file
    "../data/text_files/c.txt": """C is a low-level, compiled programming language.
- Provides direct memory access and system-level control
- Foundation for many modern languages (C++, Java, Python)
- Highly efficient and fast execution
- Requires manual memory management
- Portable across different hardware platforms
- Used in operating systems, embedded systems, drivers
- Rich set of operators and control structures""",
}

In [5]:
for filepath, content in sample_texts.items():
    with open(filepath, "w") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


In [6]:
# TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python.txt", encoding="utf-8")
documents = loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms')]


In [7]:
# DirectoryLoader
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader(
    "../data/text_files", 
    glob="*.txt", 
    loader_cls=TextLoader, 
    loader_kwargs={"encoding": "utf-8"}, 
    show_progress=False
)
documents = loader.load()
print(documents)    

[Document(metadata={'source': '../data/text_files/c.txt'}, page_content='C is a low-level, compiled programming language.\n- Provides direct memory access and system-level control\n- Foundation for many modern languages (C++, Java, Python)\n- Highly efficient and fast execution\n- Requires manual memory management\n- Portable across different hardware platforms\n- Used in operating systems, embedded systems, drivers\n- Rich set of operators and control structures'), Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms')]


### Chunking

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [10]:
chunks=split_documents(documents)
chunks

Split 2 documents into 2 chunks

Example chunk:
Content: C is a low-level, compiled programming language.
- Provides direct memory access and system-level control
- Foundation for many modern languages (C++, Java, Python)
- Highly efficient and fast executi...
Metadata: {'source': '../data/text_files/c.txt'}


[Document(metadata={'source': '../data/text_files/c.txt'}, page_content='C is a low-level, compiled programming language.\n- Provides direct memory access and system-level control\n- Foundation for many modern languages (C++, Java, Python)\n- Highly efficient and fast execution\n- Requires manual memory management\n- Portable across different hardware platforms\n- Used in operating systems, embedded systems, drivers\n- Rich set of operators and control structures'),
 Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms')]

### Embedding

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model '{self.model_name}'...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, text: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded.")
                             
        print(f"Generating embeddings for {len(text)} texts...")
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("Embeddings generated successfully.")
        return embeddings

In [13]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading model 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [14]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "Collection of PDF document embeddings"})

            print(f"Vector store initialized successfully. Collection name: '{self.collection_name}' Existing document at collection '{self.collection.count()}'")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        # Prepare data for chromadb
        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embedding_list
            )
            print(f"Added {len(documents)} documents and embeddings to the vector store successfully.")
        except Exception as e:
            print(f"Error adding embeddings to vector store: {e}")
            raise

In [15]:
vector_store = VectorStore()
vector_store

Vector store initialized successfully. Collection name: 'pdf_documents' Existing document at collection '2'


### Convert the texts to embeddings

In [16]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 2 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.


### Store embeddings into vector store

In [17]:
vector_store.add_documents(chunks, embeddings)

Added 2 documents and embeddings to the vector store successfully.


### Rag Retriever

In [18]:
class RAGRetriever:

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        print(f"Retrieving relevant documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} relevant documents for the query: '{query}'")
                return retrieved_docs
            
            else:
                print(f"Failed to retrieve results for query: '{query}'")


        except Exception as e:
            print(f"Error querying vector store: {e}")
            raise

In [19]:
reg_retriever = RAGRetriever(vector_store, embedding_manager)
reg_retriever

### Query

In [20]:
reg_retriever.retrieve("What are the key feature of python?")

Retrieving relevant documents for query: 'What are the key feature of python?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.
Retrieved 2 relevant documents for the query: 'What are the key feature of python?'


[{'id': 'doc_e3582611_1',
  'content': 'Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large community and excellent documentation\n- Supports multiple programming paradigms',
  'metadata': {'doc_index': 1,
   'source': '../data/text_files/python.txt',
   'content_length': 394},
  'similarity_score': 0.2604427933692932,
  'distance': 0.7395572066307068,
  'rank': 1},
 {'id': 'doc_846d1d9e_1',
  'content': 'Python is a high-level, interpreted programming language.\n- Easy to learn and read with clean syntax\n- Versatile: web development, data science, AI, automation\n- Extensive standard library and third-party packages\n- Dynamic typing with strong type inference\n- Cross-platform support (Windows, Linux, macOS)\n- Large c

### Integration VectorDB Context Pipeline with LLM Output